# 15 — Price co-movement

Estimate Portugal-Spain retail price co-movement using weekly pre-tax prices. The notebook requires a tidy price-history extraction and persists coefficients and diagnostics.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.prices import choose_price_model, fit_price_comovement, fit_short_run_price_transmission, price_comovement_design, spread_stationarity, stationarity_diagnostics


## Required tidy input contract

Create `data/interim/weekly_oil_prices_tidy.csv` from the Commission workbook with:

`date, country, product, price_with_tax_eur_per_1000l, price_without_tax_eur_per_1000l`

for at least `PT` and `ES`, and products `diesel` and `gasoline`.

The source workbook is deliberately inventoried in notebook 05 because historical sheet layouts can change. Never silently guess columns if the workbook changes.


In [ ]:
path = PATHS.interim / "weekly_oil_prices_tidy.csv"
if not path.exists():
    raise FileNotFoundError(
        "Missing data/interim/weekly_oil_prices_tidy.csv. Extract it from the audited EC workbook before running price models."
    )
prices = pd.read_csv(path, parse_dates=["date"])
required = {"date", "country", "product", "price_without_tax_eur_per_1000l"}
if not required.issubset(prices.columns):
    raise ValueError(f"Price input missing: {sorted(required - set(prices.columns))}")


In [ ]:
stationarity = stationarity_diagnostics(
    prices,
    value_column="price_without_tax_eur_per_1000l",
    group_columns=["country", "product"],
)
persist_dataframe(stationarity, PATHS.metrics / "price_stationarity_diagnostics.csv")
display(stationarity)


In [ ]:
# PT pre-tax price on Spain pre-tax price, with a post-transition interaction for changed co-movement.
rows = []
choice_rows = []
spread_rows = []
short_run_rows = []
for product in ["diesel", "gasoline"]:
    wide = price_comovement_design(prices, product=product)
    choice = choose_price_model(wide, product=product)
    choice_rows.append(choice)
    # The levels regression is persisted either way, but never without the verdict
    # that says whether levels are admissible. Anything other than "levels" means
    # the short-run log-difference model below carries the inference.
    family = str(choice["model_family"])
    levels_valid = family == "levels"
    spread_rows.append({"product": product, **spread_stationarity(wide)})
    model = fit_price_comovement(wide)
    for term in model.params.index:
        rows.append({"product": product, "term": term, "estimate": model.params[term], "std_error": model.bse[term], "p_value": model.pvalues[term], "nobs": model.nobs, "covariance": "HAC(8)", "outcome": "PT pre-tax EUR/1000L", "comparison": "ES pre-tax EUR/1000L", "model": "PT-ES price co-movement with ES_x_post interaction", "model_family": family, "levels_model_valid": levels_valid})
    short_model = fit_short_run_price_transmission(wide)
    for term in short_model.params.index:
        short_run_rows.append({"product": product, "term": term, "estimate": short_model.params[term], "std_error": short_model.bse[term], "p_value": short_model.pvalues[term], "nobs": short_model.nobs, "covariance": "HAC(8)", "outcome": "delta log PT pre-tax price", "comparison": "delta log ES pre-tax price", "model": "short-run log-difference transmission", "model_family": family})
coefs = pd.DataFrame(rows)
persist_dataframe(coefs, PATHS.metrics / "price_comovement_models.csv")
choices = pd.DataFrame(choice_rows)
persist_dataframe(choices, PATHS.metrics / "price_model_choice.csv", key_columns=["product"])
spread_tests = pd.DataFrame(spread_rows)
persist_dataframe(spread_tests, PATHS.metrics / "pt_es_spread_stationarity.csv", key_columns=["product", "diagnostic"])
short_run = pd.DataFrame(short_run_rows)
persist_dataframe(short_run, PATHS.metrics / "price_short_run_models.csv")
display(coefs)
